In [18]:
from pathlib import Path
import pandas as pd
from cyvcf2 import VCF
from tqdm import tqdm

In [19]:
pd.set_option('display.max_columns', None)

In [20]:
snp_only_dir = Path("./snp_only")

vcf_files = sorted(snp_only_dir.glob("*.vcf"))

In [21]:
# Collect all unique SNP sites across samples 
all_sites = set()
sample_sites = {}

print("🔍 Reading SNP sites from all VCFs...")
for vcf_file in tqdm(vcf_files):
    sample_name = vcf_file.stem.replace("_snps_only", "")
    reader = VCF(str(vcf_file))
    sites = []
    for record in reader:
        # Use (CHROM, POS, REF, ALT) tuple for uniqueness
        for alt in record.ALT:
            site = (record.CHROM, record.POS, record.REF, str(alt))
            sites.append(site)
            all_sites.add(site)
    sample_sites[sample_name] = sites

all_sites = sorted(all_sites)

print(f"✅ Total unique SNP sites: {len(all_sites)}")

🔍 Reading SNP sites from all VCFs...


100%|██████████| 400/400 [04:00<00:00,  1.66it/s] 

✅ Total unique SNP sites: 1710


In [22]:
matrix = []
for sample_name, sites in tqdm(sample_sites.items(), desc="🧬 Building SNP matrix"):
    sites_set = set(sites)
    row = [1 if site in sites_set else 0 for site in all_sites]
    matrix.append(row)

columns = [f"{chrom}_{pos}_{ref}_{alt}" for chrom, pos, ref, alt in all_sites]
df = pd.DataFrame(matrix, columns=columns, index=sample_sites.keys())

print("✅ SNP Matrix shape:", df.shape)

🧬 Building SNP matrix:   0%|          | 0/400 [00:00<?, ?it/s]

🧬 Building SNP matrix: 100%|██████████| 400/400 [00:00<00:00, 3551.05it/s]


✅ SNP Matrix shape: (400, 1710)


In [23]:
df.to_csv("snp_matrix.csv")
print("📁 Saved SNP matrix to snp_matrix.csv")

📁 Saved SNP matrix to snp_matrix.csv


In [24]:
df

Chromosome_71_C_T  Chromosome_82_G_C  Chromosome_195_G_C  \
ERR038257.vcf                   0                  0                   0   
ERR046771.vcf                   0                  0                   0   
ERR046840.vcf                   0                  0                   0   
ERR046847.vcf                   0                  0                   0   
ERR046937.vcf                   0                  0                   0   
...                           ...                ...                 ...   
ERR4831907.vcf                  0                  0                   0   
ERR4831952.vcf                  0                  0                   0   
ERR4831955.vcf                  0                  0                   0   
ERR4872232.vcf                  0                  0                   0   
ERR550776.vcf                   0                  0                   0   

                Chromosome_252_C_A  Chromosome_266_G_A  Chromosome_290_G_C  \
ERR038257.vcf                    0                   0                   0   
ERR046771.vcf                    0                   0                   0   
ERR046840.vcf                    0                   0                   0   
ERR046847.vcf                    0                   0                   0   
ERR046937.vcf                    0                   0                   0   
...                            ...                 ...                 ...   
ERR4831907.vcf                   0                   0                   0   
ERR4831952.vcf                   0                   0                   0   
ERR4831955.vcf                   0                   0                   0   
ERR4872232.vcf                   0                   0                   0   
ERR550776.vcf                    0                   0                   0   

                Chromosome_309_G_C  Chromosome_371_C_T  Chromosome_467_A_G  \
ERR038257.vcf                    0                   0                   0   
ERR046771.vcf                    0                   0                   0   
ERR046840.vcf                    0                   0                   0   
ERR046847.vcf                    0                   0                   0   
ERR046937.vcf                    0                   0                   0   
...                            ...                 ...                 ...   
ERR4831907.vcf                   0                   0                   0   
ERR4831952.vcf                   0                   0                   0   
ERR4831955.vcf                   0                   0                   0   
ERR4872232.vcf                   0                   0                   0   
ERR550776.vcf                    0                   0                   0   

                Chromosome_485_C_T  Chromosome_526_A_G  Chromosome_645_A_C  \
ERR038257.vcf                    0                   0                   0   
ERR046771.vcf                    0                   0                   0   
ERR046840.vcf                    0                   0                   0   
ERR046847.vcf                    0                   0                   0   
ERR046937.vcf                    0                   0                   0   
...                            ...                 ...                 ...   
ERR4831907.vcf                   0                   0                   0   
ERR4831952.vcf                   0                   0                   0   
ERR4831955.vcf                   0                   0                   0   
ERR4872232.vcf                   0                   0                   0   
ERR550776.vcf                    0                   0                   0   

                Chromosome_672_T_C  Chromosome_697_C_T  Chromosome_698_G_A  \
ERR038257.vcf                    0                   0                   0   
ERR046771.vcf                    0                   0                   0   
ERR046840.vcf                    0                   0    

# Filtering based on Minor Allele Frequency (MAF)

## What is MAF?

| Aspect                | Minor Allele Frequency (MAF)                        |
| --------------------- | --------------------------------------------------- |
| **Definition**        | Frequency of less common allele in population       |
| **Range**             | 0 to 0.5                                            |
| **Why filter it?**    | To remove rare SNPs that are uninformative or noisy |
| **Common thresholds** | MAF ≥ 0.01 or MAF ≥ 0.05                            |
| **Used in**           | GWAS, machine learning, population genetics, QC     |


## Approaches

| Aspect                    | First Code (`threshold = 5`)     | Second Code (`minimum_frequency = 0.05`)   |
| ------------------------- | -------------------------------- | ------------------------------------------ |
| **Filtering Based On**    | Absolute count of samples        | Proportion of samples                      |
| **Formula**               | SNP kept if found in ≥ 5 samples | SNP kept if found in ≥ 5% of samples       |
| **More Flexible?**        | No, fixed threshold              | Yes, scales with sample size               |
| **Example**               | Keeps SNPs in ≥ 5 samples        | Keeps SNPs in ≥ `0.05 * 251 = ~20` samples |

In [31]:
# Count how many samples have each SNP (i.e., how many 1s per column)
snp_counts = df.sum(axis=0)
snp_counts

Chromosome_71_C_T         2
Chromosome_82_G_C         1
Chromosome_195_G_C        1
Chromosome_252_C_A        1
Chromosome_266_G_A        1
                         ..
Chromosome_4408358_G_C    1
Chromosome_4408413_G_A    1
Chromosome_4408430_C_T    1
Chromosome_4408456_G_C    1
Chromosome_4408459_C_T    1
Length: 1710, dtype: int64

## MAF based on sample counts

In [33]:
maf_sc_df = df.copy()
print("🔢 Initial matrix shape:", maf_sc_df.shape)

🔢 Initial matrix shape: (400, 1710)


In [34]:
# Filter threshold: keep SNPs present in >= 5 samples
threshold = 5
maf_sc_df = maf_sc_df.loc[:, snp_counts >= threshold]

print("🔢 Matrix shape after filtering rare SNPs with a 5 threshold:", maf_sc_df.shape)

🔢 Matrix shape after filtering rare SNPs with a 5 threshold: (400, 226)


In [35]:
maf_sc_df

,Chromosome_371_C_T,Chromosome_5075_C_T,Chromosome_5520_C_T,Chromosome_6112_G_C,Chromosome_6124_C_T,Chromosome_6446_G_T,Chromosome_7222_C_T,Chromosome_7268_C_T,Chromosome_7362_G_C,Chromosome_7570_C_T,Chromosome_7581_G_T,Chromosome_7582_A_C,Chromosome_7582_A_G,Chromosome_7585_G_C,Chromosome_7892_G_A,Chromosome_8040_G_A,Chromosome_8452_C_T,Chromosome_8688_G_T,Chromosome_9143_T_C,Chromosome_9260_G_C,Chromosome_9304_G_A,Chromosome_13298_G_C,Chromosome_13460_A_G,Chromosome_13482_G_A,Chromosome_13699_T_C,Chromosome_491290_G_A,Chromosome_491591_A_T,Chromosome_491742_T_C,Chromosome_575679_A_G,Chromosome_575907_C_T,Chromosome_576077_C_T,Chromosome_620029_C_T,Chromosome_620553_T_C,Chromosome_620625_A_G,Chromosome_657081_C_T,Chromosome_657142_C_T,Chromosome_657269_A_G,Chromosome_657425_C_T,Chromosome_657578_A_G,Chromosome_734116_T_C,Chromosome_759546_A_G,Chromosome_759746_C_T,Chromosome_760115_C_T,Chromosome_760490_C_T,Chromosome_761110_A_T,Chromosome_761139_C_T,Chromosome_761155_C_T,Chromosome_761489_G_A,Chromosome_762089_G_C,Chromosome_762434_T_G,Chromosome_763031_T_C,Chromosome_763884_C_T,Chromosome_763886_C_A,Chromosome_764817_T_C,Chromosome_764817_T_G,Chromosome_764840_A_G,Chromosome_764995_C_G,Chromosome_765150_G_A,Chromosome_765171_C_T,Chromosome_766488_C_G,Chromosome_766645_A_C,Chromosome_775639_T_C,Chromosome_776100_G_A,Chromosome_776182_C_T,Chromosome_776395_A_G,Chromosome_778298_C_T,Chromosome_779615_G_C,Chromosome_781276_A_C,Chromosome_781395_T_C,Chromosome_781687_A_G,Chromosome_781822_A_G,Chromosome_800204_T_C,Chromosome_800219_T_C,Chromosome_800357_C_A,Chromosome_1254562_A_G,Chromosome_1302899_A_G,Chromosome_1303601_A_G,Chromosome_1364706_G_A,Chromosome_1417019_C_T,Chromosome_1417554_G_C,Chromosome_1417793_G_A,Chromosome_1471659_C_T,Chromosome_1472337_C_T,Chromosome_1472359_A_C,Chromosome_1472362_C_T,Chromosome_1473246_A_G,Chromosome_1474001_C_T,Chromosome_1673425_C_T,Chromosome_1834177_A_C,Chromosome_1834836_T_C,Chromosome_1853974_C_T,Chromosome_1854045_A_G,Chromosome_1854169_A_C,Chromosome_1854300_T_C,Chromosome_1917972_A_G,Chromosome_2062922_T_C,Chromosome_2063187_G_A,Chromosome_2063685_C_T,Chromosome_2063911_A_G,Chromosome_2102240_C_T,Chromosome_2102990_A_G,Chromosome_2154724_C_A,Chromosome_2155168_C_G,Chromosome_2167926_A_G,Chromosome_2167983_C_T,Chromosome_2168149_G_A,Chromosome_2168604_G_A,Chromosome_2168742_C_T,Chromosome_2169840_C_T,Chromosome_2170568_G_C,Chromosome_2222308_T_C,Chromosome_2223293_T_C,Chromosome_2223372_G_C,Chromosome_2223553_G_T,Chromosome_2288847_C_G,Chromosome_2289047_G_A,Chromosome_2289162_A_G,Chromosome_2518076_C_T,Chromosome_2518132_C_T,Chromosome_2518919_G_A,Chromosome_2714846_C_T,Chromosome_2715369_C_A,Chromosome_2726051_G_A,Chromosome_2726105_G_A,Chromosome_2782498_G_A,Chromosome_2783921_C_T,Chromosome_2784617_A_G,Chromosome_2995848_G_A,Chromosome_2996208_C_A,Chromosome_2996912_C_T,Chromosome_3064632_G_A,Chromosome_3066099_C_T,Chromosome_3066280_G_A,Chromosome_3067039_A_G,Chromosome_3073868_T_C,Chromosome_3086731_A_G,Chromosome_3086742_A_C,Chromosome_3086788_T_C,Chromosome_3339417_A_G,Chromosome_3448497_T_A,Chromosome_3448567_C_G,Chromosome_3448608_G_A,Chromosome_3448714_G_C,Chromosome_3569220_G_A,Chromosome_3612009_C_T,Chromosome_3612813_T_C,Chromosome_3624350_C_A,Chromosome_3624486_T_C,Chromosome_3625065_T_G,Chromosome_3626467_C_T,Chromosome_3626562_G_A,Chromosome_3640211_T_G,Chromosome_3640557_T_C,Chromosome_3641447_C_T,Chromosome_3642877_A_G,Chromosome_3840764_C_G,Chromosome_3841473_C_T,Chromosome_3877553_C_T,Chromosome_4038287_G_A,Chromosome_4038318_G_A,Chromosome_4040517_A_G,Chromosome_4040719_T_C,Chromosome_4044740_G_A,Chromosome_4044872_G_A,Chromosome_4045645_C_T,Chromosome_4046007_C_T,Chromosome_4138377_A_G,Chromosome_4138622_G_A,Chromosome_4139246_G_A,Chromosome_4238120_G_A,Chromosome_4238675_C_T,Chromosome_4238963_C_T,Chromosome_4239298_C_T,Chromosome_4240671_C_T,Chromosome_4240897_C_G,Chromosome_4241022_C_T,Chromosome_4241042_A_G,Chromosome_4242075_G_A,Chromosome_4242182_G_T,Chr

In [36]:
df.to_csv("maf_sc_snp_matrix.csv")
print("📁 Saved SNP matrix to maf_sc_snp_matrix.csv")

📁 Saved SNP matrix to maf_sc_snp_matrix.csv


## MAF based on proportion of samples

In [39]:
maf_prop_df = df.copy()
print("🔢 Matrix shape:", maf_prop_df.shape)

🔢 Matrix shape: (400, 1710)


In [40]:
minimum_frequency = 0.05  # keep SNPs in at least 5% of samples

# Calculate allele frequency for each SNP (column)
allele_frequencies = maf_prop_df.sum(axis=0) / maf_prop_df.shape[0]

# Filter columns to keep only SNPs where frequency >= minimum_frequency
maf_prop_df = maf_prop_df.loc[:, allele_frequencies >= minimum_frequency]

print(f"Filtered SNP matrix shape: {maf_prop_df.shape}")

maf_prop_df

Filtered SNP matrix shape: (400, 101)


,Chromosome_6112_G_C,Chromosome_7362_G_C,Chromosome_7582_A_G,Chromosome_7585_G_C,Chromosome_8452_C_T,Chromosome_9143_T_C,Chromosome_9304_G_A,Chromosome_13298_G_C,Chromosome_13460_A_G,Chromosome_491591_A_T,Chromosome_491742_T_C,Chromosome_575679_A_G,Chromosome_575907_C_T,Chromosome_620625_A_G,Chromosome_657081_C_T,Chromosome_657142_C_T,Chromosome_657269_A_G,Chromosome_657578_A_G,Chromosome_734116_T_C,Chromosome_759746_C_T,Chromosome_760115_C_T,Chromosome_761155_C_T,Chromosome_762434_T_G,Chromosome_763031_T_C,Chromosome_763884_C_T,Chromosome_763886_C_A,Chromosome_764995_C_G,Chromosome_765150_G_A,Chromosome_765171_C_T,Chromosome_766645_A_C,Chromosome_775639_T_C,Chromosome_776100_G_A,Chromosome_776182_C_T,Chromosome_779615_G_C,Chromosome_781395_T_C,Chromosome_781687_A_G,Chromosome_800219_T_C,Chromosome_800357_C_A,Chromosome_1254562_A_G,Chromosome_1364706_G_A,Chromosome_1417019_C_T,Chromosome_1417554_G_C,Chromosome_1417793_G_A,Chromosome_1471659_C_T,Chromosome_1474001_C_T,Chromosome_1673425_C_T,Chromosome_1834177_A_C,Chromosome_1853974_C_T,Chromosome_1854045_A_G,Chromosome_1854300_T_C,Chromosome_1917972_A_G,Chromosome_2062922_T_C,Chromosome_2154724_C_A,Chromosome_2155168_C_G,Chromosome_2167926_A_G,Chromosome_2167983_C_T,Chromosome_2222308_T_C,Chromosome_2223293_T_C,Chromosome_2289047_G_A,Chromosome_2518076_C_T,Chromosome_2518132_C_T,Chromosome_2726051_G_A,Chromosome_2726105_G_A,Chromosome_2782498_G_A,Chromosome_3064632_G_A,Chromosome_3073868_T_C,Chromosome_3086788_T_C,Chromosome_3448714_G_C,Chromosome_3612813_T_C,Chromosome_3624486_T_C,Chromosome_3625065_T_G,Chromosome_3626562_G_A,Chromosome_4038287_G_A,Chromosome_4040517_A_G,Chromosome_4044872_G_A,Chromosome_4046007_C_T,Chromosome_4138377_A_G,Chromosome_4138622_G_A,Chromosome_4238675_C_T,Chromosome_4239298_C_T,Chromosome_4240671_C_T,Chromosome_4241042_A_G,Chromosome_4242075_G_A,Chromosome_4242643_C_T,Chromosome_4242803_G_C,Chromosome_4243460_C_T,Chromosome_4245969_C_T,Chromosome_4247429_A_G,Chromosome_4247646_A_C,Chromosome_4249408_G_A,Chromosome_4267647_T_C,Chromosome_4269387_T_G,Chromosome_4269606_A_G,Chromosome_4328492_G_A,Chromosome_4329782_G_A,Chromosome_4338603_G_A,Chromosome_4338732_G_A,Chromosome_4407588_T_C,Chromosome_4407873_C_A,Chromosome_4407927_T_G,Chromosome_4408156_A_C
ERR038257.vcf,0,1,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,1,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0
ERR046771.vcf,0,1,0,1,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,1,1,0,0,0,0,0,0,1,1,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,0,1,1,0,1,0,1,0,0,1,1,0,0,0,1,1,0,0,1,0,0,0,1,0,0,0,1,0,0,1,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,1,1,0,1,1,0,0,0
ERR046840.vcf,0,1,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,1,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0
ERR046847.vcf,0,1,0,1,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,1,1,0,0,0,0,0,0,1,1,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,0,1,1,0,1,0,1,0,0,1,1,0,0,0,1,1,0,0,1,0,0,0,1,0,0,0,1,0,0,1,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,1,1,0,1,1,0,0,0
ERR046937.vcf,0,1,0,1,0,0,1,0,0,1,0,1,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,1,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ERR4831907.vcf,0,1,0,1,0,0,1,0,0,0,1,0,1,1,1,1,0,0,0,0,0,1,0,1,0,0,0,0,0,0,1,1,1,1,1,1,0,0,1,0,0,0,0,1,0,0,1,0,0,1,1,0,1,1,1,0,0,1,0,0,0,0,0,0,0,0,1,0,1,0,1,1,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,1,0,0,1,0,0,0,1,0,1,1,0,1,0
ERR4831952.vcf,0,1,0,1,0,

In [37]:
df.to_csv("maf_prop_snp_matrix.csv")
print("📁 Saved SNP matrix to maf_prop_snp_matrix.csv")

📁 Saved SNP matrix to maf_prop_snp_matrix.csv
